# NoteTriage Colab demo

Classify a fictional IT discovery note as a **fact**, **assumption**, or **open question**. Every fact needs a verbatim source quote.

This notebook is for an ad-hoc demo only. It does not change or evaluate the frozen final holdout.

In [ ]:
# Clone the project and make its source package importable.
!git clone --depth 1 https://github.com/huluobo-ovo/note-triage.git
%cd note-triage
import sys
sys.path.insert(0, 'src')


In [ ]:
# Enter your OpenRouter credential locally for this Colab session.
# The key is never written into this notebook or committed to Git.
import os
from getpass import getpass

os.environ['OPENROUTER_API_KEY'] = getpass('OpenRouter API key: ')
os.environ['OPENROUTER_MODEL'] = 'openai/gpt-4o-2024-08-06'
print('Model configured:', os.environ['OPENROUTER_MODEL'])


## Paste a demo note

Put **one sentence per line**. Use fictional or anonymised content only.

In [ ]:
demo_note = '''
Client confirmed staff use the corporate directory to sign in.
The existing VPN should be sufficient for the vendor integration.
Who approves the new firewall rule?
'''

print(demo_note)


In [ ]:
from note_triage.baseline import classify_all
from note_triage.openrouter import classify
from note_triage.types import Sentence
from note_triage.validation import validate_prediction

lines = [line.strip() for line in demo_note.splitlines() if line.strip()]
sentences = [Sentence('DEMO', index, text) for index, text in enumerate(lines, start=1)]

baseline = classify_all(sentences)
model_run = classify(sentences)
gpt = [validate_prediction(sentence, prediction)
       for sentence, prediction in zip(sentences, model_run.predictions, strict=True)]


In [ ]:
# Present a side-by-side result table.
import pandas as pd

rows = []
for sentence, rule, llm in zip(sentences, baseline, gpt, strict=True):
    rows.append({
        'Sentence': sentence.text,
        'Rule baseline': rule.label.value,
        'GPT-4o': llm.label.value,
        'GPT-4o fact quote': llm.quote or '—',
    })

display(pd.DataFrame(rows))
print(f'Latency: {model_run.latency_seconds:.2f}s')
print('Prompt tokens:', model_run.prompt_tokens)
print('Completion tokens:', model_run.completion_tokens)


## Interpretation

- A missing or non-verbatim fact quote is automatically downgraded to `abstain`.
- The output is decision support: a person must review it before it becomes a delivery commitment.
- Saved final results are in `evals/results/gpt4o_holdout_run.json`; do not rerun or tune against the frozen holdout.